# 01 · Training on Google Colab (GPU)
**4-Modal Fusion System — Full Training Pipeline**

> Run this notebook on **Google Colab** with GPU runtime
> `Runtime → Change runtime type → T4 GPU`

### Workflow
1. Check GPU
2. Mount Google Drive
3. Install dependencies
4. Upload / point to fusion_system package
5. Verify data access
6. Run group k-fold training
7. DANN lambda sweep (paper ablation)
8. Save artifacts to Drive

## 1 · Environment Check

In [ ]:
import subprocess, sys, platform, os

print("Python  :", sys.version)
print("Platform:", platform.platform())

# GPU check
try:
    import torch
    if torch.cuda.is_available():
        dev = torch.cuda.get_device_properties(0)
        print(f"\nGPU     : {dev.name}")
        print(f"VRAM    : {dev.total_memory / 1e9:.1f} GB")
        print(f"CUDA    : {torch.version.cuda}")
        DEVICE  = torch.device("cuda")
    else:
        print("\nNo CUDA GPU found — falling back to CPU")
        DEVICE  = torch.device("cpu")
except ImportError:
    print("torch not installed yet — run cell 3 first")
    DEVICE = None

# Colab detection
IN_COLAB = "google.colab" in sys.modules
print(f"\nColab   : {IN_COLAB}")

## 2 · Mount Google Drive

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive/fusion_project"
    import os; os.makedirs(DRIVE_ROOT, exist_ok=True)
    print("Drive mounted at:", DRIVE_ROOT)
else:
    # Local run
    from pathlib import Path
    DRIVE_ROOT = str(Path.cwd().parent)
    print("Local mode — DRIVE_ROOT:", DRIVE_ROOT)

## 3 · Install Dependencies

In [ ]:
# Run this cell once per Colab session
import subprocess, sys

pkgs = [
    "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121",
    "timm",
    "numpy pandas pyarrow scikit-learn scipy Pillow",
    "matplotlib seaborn tqdm",
    "mlflow joblib",
]

for pkg in pkgs:
    print(f"Installing: {pkg[:60]}...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pkg.split(),
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print("  WARN:", result.stderr[-200:])
    else:
        print("  OK")

print("\nAll packages installed.")

## 4 · Fusion System Path Setup

In [ ]:
import sys, os
from pathlib import Path

# ── Option A: Colab — upload fusion_system as a zip to Drive ──
# FUSION_DIR = "/content/drive/MyDrive/fusion_project/fusion_system"

# ── Option B: Clone from private repo (replace with your URL) ──
# !git clone https://github.com/YOUR_ORG/fusion_system /content/fusion_system
# FUSION_DIR = "/content/fusion_system"

# ── Option C: Colab — notebook is inside fusion_system/notebooks/ ──
NB_DIR     = Path.cwd()
FUSION_DIR = NB_DIR.parent if (NB_DIR.parent / "config.py").exists() else Path(DRIVE_ROOT)

sys.path.insert(0, str(FUSION_DIR))
print("FUSION_DIR:", FUSION_DIR)

import config as C
print("config.py loaded — N_LABELS:", C.N_LABELS)

# Override paths if running from Drive
import config as C_mod
DATA_ON_DRIVE = Path(DRIVE_ROOT) / "dataset"
if DATA_ON_DRIVE.exists():
    C_mod.DATA_ROOT       = DATA_ON_DRIVE
    C_mod.PARQUET_PATH    = DATA_ON_DRIVE / "dataset_V2.parquet"
    C_mod.THERMAL_CSV_DIR = DATA_ON_DRIVE / "Thermographie" / "Thermographie"
    C_mod.CV_IMAGE_DIR    = DATA_ON_DRIVE / "CV_images_Vorversuche_PP" / "Rohbilder"
    print("Using Drive dataset:", C_mod.PARQUET_PATH)
else:
    print("Using config.py paths (local/default)")

C_mod.MODELS_DIR    = Path(DRIVE_ROOT) / "saved_models"
C_mod.ARTIFACTS_DIR = Path(DRIVE_ROOT) / "artifacts"
C_mod.MODELS_DIR.mkdir(parents=True, exist_ok=True)
C_mod.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print("Models dir    :", C_mod.MODELS_DIR)
print("Artifacts dir :", C_mod.ARTIFACTS_DIR)

## 5 · Data Verification

In [ ]:
from data.loaders import load_parquet, build_cv_index, join_cv_to_parquet
from data.dataset import get_tabular_cols, get_dxp_cols
from training.splits import build_experiment_label_map
import pandas as pd

print("Loading parquet...")
df = load_parquet(C.PARQUET_PATH)
print(f"  Shape          : {df.shape}")
print(f"  Parts          : {len(df)}")
print(f"  Experiments    : {df[C.GROUP_COL].nunique()}")

print("\nBuilding CV index...")
cv_idx = build_cv_index()
df     = join_cv_to_parquet(df, cv_idx)
print(f"  CV-joined shape: {df.shape}")

tabular_cols  = get_tabular_cols(df)
dxp_cols      = get_dxp_cols(df)
exp_label_map = build_experiment_label_map(df)
n_experiments = len(exp_label_map)

has_thermal = df["has_thermal"].sum() if "has_thermal" in df.columns else 0
has_visual  = df["has_visual"].sum()  if "has_visual"  in df.columns else 0

print(f"\n  Tabular cols   : {len(tabular_cols)}")
print(f"  DXP channels   : {len(dxp_cols)}")
print(f"  Experiments    : {n_experiments}")
print(f"  Has thermal    : {has_thermal}/{len(df)}")
print(f"  Has visual     : {has_visual}/{len(df)}")
print(f"\nLabel distribution:")
print(df[C.LABEL_COLS].sum().to_string())

## 6 · Model Instantiation

In [ ]:
import torch
from models.fusion import InjectionMouldingFusionModel

model = InjectionMouldingFusionModel(
    tabular_in_dim = len(tabular_cols),
    n_dxp_channels = len(dxp_cols),
    n_experiments  = n_experiments,
    dann_lambda    = 0.0,
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 55)
print("MODEL SUMMARY")
print("=" * 55)
print(f"  Total parameters     : {total_params:,}")
print(f"  Trainable parameters : {trainable_params:,}")
print(f"  Device               : {DEVICE}")
print(f"  DANN head            : {n_experiments} experiment classes")
print(f"  Defect head          : {C.N_LABELS} labels")
print("=" * 55)

# Sub-module parameter counts
for name, mod in [("ThermalEncoder", model.thermal_enc),
                  ("VisualEncoder",  model.visual_enc),
                  ("TCNEncoder",     model.tcn_enc),
                  ("TabularEncoder", model.tab_enc),
                  ("CrossModalFusion",model.fusion),
                  ("DefectHead",     model.defect_head)]:
    n = sum(p.numel() for p in mod.parameters())
    print(f"  {name:22s}: {n:>9,} params")

## 7 · Group K-Fold Training (HEADLINE)

In [ ]:
import time, json
import numpy as np
from training.trainer import train_one_fold, run_epoch
from training.splits  import group_kfold_splits
from training.losses  import TotalLoss
from training.metrics import compute_metrics, find_best_thresholds, group_aware_evaluation
from torch.utils.data import DataLoader
from data.dataset import MultiModalDataset, collate_fn
from sklearn.preprocessing import StandardScaler
from data.loaders import load_tabular_vector

fold_results   = []
all_val_probs  = []
all_val_targets= []
all_val_groups = []

print(f"Starting {C.N_GROUP_FOLDS}-fold group cross-validation...")
print(f"Epochs: {C.NUM_EPOCHS}  |  Batch: {C.BATCH_SIZE}  |  Device: {DEVICE}")
print("=" * 55)

for fold_i, (train_df, val_df) in enumerate(group_kfold_splits(df)):
    print(f"\n=== FOLD {fold_i+1}/{C.N_GROUP_FOLDS}  ===")
    print(f"  Train: {len(train_df)} parts  |  Val: {len(val_df)} parts")

    # Fresh model each fold
    fold_model = InjectionMouldingFusionModel(
        tabular_in_dim=len(tabular_cols),
        n_dxp_channels=len(dxp_cols),
        n_experiments=n_experiments,
    ).to(DEVICE)

    t0 = time.time()
    fold_out = train_one_fold(
        fold_model, train_df, val_df,
        tabular_cols, dxp_cols, exp_label_map, DEVICE,
        fold_tag=f"fold{fold_i+1}",
    )

    # Inference on validation fold for aggregation
    scaler = StandardScaler()
    tab_vals = [load_tabular_vector(row, tabular_cols) for _, row in train_df.iterrows()]
    scaler.fit(np.array(tab_vals))
    tab_med = {c: float(train_df[c].median()) for c in tabular_cols if c in train_df.columns}
    roi_med = {c: float(train_df[c].median()) for c in C.THERMAL_ROI_COLS if c in train_df.columns}

    val_ds = MultiModalDataset(val_df, tabular_cols, dxp_cols, train=False,
                               tabular_medians=tab_med, roi_medians=roi_med,
                               tabular_scaler=scaler)
    val_ld = DataLoader(val_ds, batch_size=C.BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn, num_workers=2)

    _, vp, vt, vg = run_epoch(fold_model, val_ld, TotalLoss(), None,
                               DEVICE, exp_label_map, train=False)
    all_val_probs.append(vp)
    all_val_targets.append(vt)
    all_val_groups.append(vg)

    elapsed = time.time() - t0
    print(f"  Best val F1 : {fold_out['best_val_f1']:.4f}  ({elapsed:.0f}s)")
    fold_results.append({"fold": fold_i+1, "val_f1": fold_out["best_val_f1"]})

# Aggregate
all_p = np.concatenate(all_val_probs,   axis=0)
all_t = np.concatenate(all_val_targets, axis=0)
all_g = np.concatenate(all_val_groups,  axis=0)

group_thrs     = find_best_thresholds(all_p, all_t)
group_metrics  = compute_metrics(all_p, all_t, thresholds=group_thrs)
group_breakdown= group_aware_evaluation(all_p, all_t, all_g, thresholds=group_thrs)

print("\n" + "=" * 55)
print("GROUP K-FOLD HEADLINE RESULTS")
print("=" * 55)
print(f"  Macro-F1 : {group_metrics['macro_f1']:.4f}")
print(f"  Micro-F1 : {group_metrics['micro_f1']:.4f}")
print(f"  ROC-AUC  : {group_metrics['mean_roc_auc']:.4f}")
print(f"  PR-AUC   : {group_metrics['mean_pr_auc']:.4f}")
print(f"  Per-group mean±std: {group_breakdown['mean_group_f1']:.4f} ± {group_breakdown['std_group_f1']:.4f}")
print("=" * 55)

## 8 · Training Curves

In [ ]:
import matplotlib.pyplot as plt

# Use last fold's history for illustration
history = fold_out["history"]
epochs  = [h["epoch"] for h in history]
val_f1s = [h["val_f1"] for h in history]
t_loss  = [h.get("total", 0) for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, val_f1s, color="#1976D2", linewidth=2, label="Val macro-F1")
ax1.axhline(max(val_f1s), ls="--", color="#43A047", label=f"Best={max(val_f1s):.4f}")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Macro-F1"); ax1.set_title(f"Validation F1 — Fold {C.N_GROUP_FOLDS}")
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, t_loss, color="#E53935", linewidth=2, label="Train total loss")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.set_title("Training Loss — Last Fold")
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle("Training Diagnostics (Fold N)", fontsize=12)
plt.tight_layout()
plt.show()

# Per-label F1 bar chart
per_f1 = group_metrics["per_label_f1"]
fig2, ax3 = plt.subplots(figsize=(11, 4))
colours = ["#E53935" if v < 0.5 else "#FF9800" if v < 0.7 else "#43A047"
           for v in per_f1.values()]
ax3.bar(range(len(per_f1)), list(per_f1.values()), color=colours, edgecolor="white")
ax3.set_xticks(range(len(per_f1)))
ax3.set_xticklabels([k.replace("LBL_","") for k in per_f1.keys()], rotation=30, ha="right")
ax3.set_ylabel("F1 Score"); ax3.set_ylim(0, 1.05)
ax3.set_title("Per-Label F1 — Group K-Fold Headline")
plt.tight_layout(); plt.show()

## 9 · Save Artifacts to Drive

In [ ]:
import torch, json, numpy as np

# Final model = last fold model (or re-train on all data)
C.MODELS_DIR.mkdir(parents=True, exist_ok=True)
torch.save(fold_model.state_dict(), C.MODELS_DIR / "fusion_final.pt")
np.save(C.MODELS_DIR / "best_thresholds.npy", group_thrs)

artifacts = {
    "group_kfold_metrics"  : group_metrics,
    "group_breakdown"      : group_breakdown,
    "fold_results"         : fold_results,
    "tabular_cols"         : tabular_cols,
    "dxp_cols"             : dxp_cols,
    "exp_label_map"        : exp_label_map,
    "n_experiments"        : n_experiments,
}
with open(C.ARTIFACTS_DIR / "training_results.json", "w") as f:
    json.dump(artifacts, f, indent=2, default=str)

print("Saved:")
print(f"  {C.MODELS_DIR / 'fusion_final.pt'}")
print(f"  {C.MODELS_DIR / 'best_thresholds.npy'}")
print(f"  {C.ARTIFACTS_DIR / 'training_results.json'}")
print(f"\nFinal group macro-F1 : {group_metrics['macro_f1']:.4f}")

## 10 · DANN Lambda Ablation (Paper Experiment)

In [ ]:
# ── Run this section separately — trains 5 full models ──────
# Uncomment and run to generate Fig. X in the paper

# DANN_LAMBDAS = [0.0, 0.1, 0.2, 0.3, 0.5]
# sweep = []
# import config as C_mod
# original_lam = C_mod.DANN_LAMBDA_MAX
#
# for lam in DANN_LAMBDAS:
#     C_mod.DANN_LAMBDA_MAX = lam
#     from training.trainer import train_full_pipeline
#     res = train_full_pipeline(C.PARQUET_PATH)
#     sweep.append({
#         "lambda"         : lam,
#         "group_f1"       : res["group_kfold_metrics"]["macro_f1"],
#         "random_f1"      : res["random_split_metrics"]["macro_f1"],
#         "gap"            : res["random_split_metrics"]["macro_f1"] - res["group_kfold_metrics"]["macro_f1"],
#     })
#     print(f"lambda={lam:.2f}  group={sweep[-1]['group_f1']:.4f}  random={sweep[-1]['random_f1']:.4f}")
#
# C_mod.DANN_LAMBDA_MAX = original_lam
#
# # Plot
# import matplotlib.pyplot as plt
# lams = [r["lambda"] for r in sweep]
# fig, ax = plt.subplots(figsize=(8,4))
# ax.plot(lams, [r["group_f1"]  for r in sweep], "o-", label="Group k-fold F1 (honest)")
# ax.plot(lams, [r["random_f1"] for r in sweep], "s--", label="Random split F1 (leaky)")
# ax.fill_between(lams, [r["group_f1"] for r in sweep], [r["random_f1"] for r in sweep],
#                 alpha=0.15, color="red", label="Leakage gap")
# ax.set_xlabel("DANN Lambda"); ax.set_ylabel("Macro-F1")
# ax.set_title("DANN Lambda Sweep: Closing the Leakage Gap")
# ax.legend(); plt.tight_layout(); plt.show()

print("DANN sweep cells are commented out.")
print("Uncomment above to run the paper ablation (needs ~5× training time).")